# Modelisation du jeu de données nettoyé

Ce notebook présente une exploration PCA en 3D, un découpage train/test/validation, un appel à une fonction de prétraitement externe, et un entraînement en boucle de plusieurs modèles. *Ce workflow doit avoir le jeux de données nettoyé en entrée*

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import re

# add root path, to import all modules
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    
from src.accidents.data import load_collisions
from src.accidents.models import plot_3d, compute_methods, fit_transform_embedding

from sklearn.decomposition import PCA
from sklearn.manifold import MDS, Isomap
from sklearn.model_selection import train_test_split
import plotly.express as px

seed = 3795

DATA_PATH = Path('../data/clean/collisions_clean.csv')
df = load_collisions(DATA_PATH)

Charge : 218,066 lignes x 52 colonnes  (collisions_clean.csv)


### Choisir la colonne *target*

In [2]:
target = 'GRAVITE_3'
# target = 'GRAVITE'

classes = df[target].unique()
c = len(classes)
print(classes)

<StringArray>
['Materiel', 'Leger', 'Grave']
Length: 3, dtype: str


### Loader les colonnes et split

In [3]:
gravite_column = next((c for c in df.columns if c.upper() == target), None)
if gravite_column is None:
    raise ValueError(f'Impossible de trouver une colonne {target} dans le jeu de données nettoyé.')

# X = toutes les colonnes sauf la cible
# Les colonnes de fuite (NB_MORTS, NB_BLESSES_*, etc.) seront retirees par preprocess_data()
feature_columns = [c for c in df.columns if c.upper() not in {'GRAVITE', 'GRAVITE_3'}]
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
if not feature_columns:
    raise ValueError(f'Aucune colonne numérique disponible pour la PCA après exclusion de {target}.')

X = df[feature_columns].copy()
y = df[gravite_column].copy()

print(f'Features disponibles : {len(feature_columns)}')
print(f'Distribution de la cible :')
print(y.value_counts())

Features disponibles : 50
Distribution de la cible :
GRAVITE_3
Materiel    170106
Leger        45912
Grave         2048
Name: count, dtype: int64


## Découpage train / validation / test

Nous préparons le jeu de données pour l'entraînement en séparant d'abord un test set, puis en divisant le reste entre entraînement et validation.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=seed
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=seed
)

print('Tailles :')
print('  X_train', X_train.shape)
print('  X_val', X_val.shape)
print('  X_test', X_test.shape)

### Check des petites classes

In [ ]:
print("Proportions de chaque classe des ensembles:")
print('  Y_train_proc', y_train.value_counts())
print('  Y_val_proc', y_val.value_counts())
print('  Y_test_proc', y_test.value_counts())

## Appel au prétraitement externe

Ce notebook invoque la fonction `preprocess_data` située dans `src.accidents.preprocessing`. Cette fonction applique :
1. Le retrait des colonnes qui fuitent la cible (`NB_MORTS`, `NB_BLESSES_*`, etc.)
2. Un target encoding sur les catégorielles à forte cardinalité (`REG_ADM`, `MRC`)
3. Un one-hot encoding sur les autres catégorielles
4. Une standardisation (z-score) de toutes les colonnes

Toutes les statistiques sont calculées uniquement sur `X_train` pour éviter toute fuite.

In [ ]:
from src.accidents.preprocessing.preprocessing import preprocess_data

X_train_proc, X_val_proc, X_test_proc = preprocess_data(X_train, y_train, X_val, X_test)

print('Prétraitement terminé. Formes finales :')
print('  X_train_proc', X_train_proc.shape)
print('  X_val_proc', X_val_proc.shape)
print('  X_test_proc', X_test_proc.shape)

In [ ]:
embeds_clean = compute_methods(X_train_proc, methods=["pca"])

for method, embed in embeds_clean.items():
    fig = plot_3d(embed, 
                  labels=y_train,   # attention : y_train maintenant, pas y
                  method_name=f"{method}", 
                  label_name=f"Gravité {method}")
    fig.show()

---

# Entrainement de modèles
Initialement, nous allons faire de l'entrainement sur toutes les colonnes avec des modèles basiques. Ensuite, nous allons appliquer PCA pour réduire les dimensions et observer le changement en qualité. Finalement, nous allons choisir quelques modèles ideaux pour optimiser les paramètres

In [ ]:
from sklearn.cluster import AgglomerativeClustering, Birch, KMeans, DBSCAN
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    confusion_matrix
)
import seaborn as sns
import matplotlib.pyplot as plt

from src.accidents.models import train_and_eval

### Boucle d'entraînement de méthodes de regroupement
#### Métriques:
- Adjusted Rand Index (ARI): Mesure la similarité entre les vrais regroupements et les regroupements prédits    `(best:1, worst:-0.5)`
- Adjusted Mutual Information (AMI): Similaire à ARI, avec l'information mutuelles des regroupements    `(best:1, worst:0)`
- Homogeneity: La pureté des regroupements comparativement aux vraies classes   `(best:1, worst:0)`
- Silhouette Score: Measure de la séparation entre les regroupements (unsupervised)    `(best:1, worst:-1)`

In [ ]:

models = {
    'KMeans': KMeans(n_clusters=c, random_state=seed),
    # 'AgglomerativeClustering': AgglomerativeClustering(n_clusters=c),
    # 'Birch': Birch(n_clusters=c),
    # 'DBSCAN': DBSCAN(),
}

results_clust = train_and_eval(models, X_train_proc, y_train, X_val_proc, y_val, task_type='clustering', verbose=True)

### Boucle d'entraînement de classifieurs

Nous définissons plusieurs classifieurs scikit-learn, puis comparons leurs performances sur le jeu de validation.

In [ ]:

models = {
    'Dummy (majorite)': DummyClassifier(strategy='most_frequent', random_state=seed),
    'Dummy (stratifie)': DummyClassifier(strategy='stratified', random_state=seed),
    'GaussianNB': GaussianNB(),
    'SVM': LinearSVC(random_state=seed),
    'Decision Tree': DecisionTreeClassifier(random_state=seed, max_depth=10),
    'Random Forest': RandomForestClassifier(random_state=seed),
    'Gradient Boosting': GradientBoostingClassifier(random_state=seed)
}

trained_models, results_class_first = train_and_eval(models, X_train_proc, y_train, X_val_proc, y_val, task_type='classification', verbose=True)

# Déséquilibre de classes et réduction de dimensionalité
### Déséquilibre
La classe "Grave" ne représente que <1% des données. Du coup, les modèles ont tendance à l'ignorer pour maximiser l'accuracy globale. On compare alors trois stratégies pour améliorer la détection des cas graves, toutes appliquées au Random Forest :

1. **Aucune** : baseline, on laisse faire le déséquilibre
2. **`class_weight='balanced'`** : on pénalise plus fortement les erreurs sur les classes rares (sans toucher aux données)
3. **SMOTE** : on génère des exemples synthétiques de la classe minoritaire par interpolation

### Réduction
Utilisant la méthode PCA, trouvé ci-haut, nous allons réduire les dimensions de X, qui est autour de 164 colonnes courament.

---

## Initialisation des stratégies


In [24]:
from imblearn.over_sampling import SMOTE

# SMOTE une seule fois
smote = SMOTE(random_state=seed)
X_tr_smote, y_tr_smote = smote.fit_resample(X_train_proc, y_train)

pca_sizes = [None, 50, 20, 10, 5, 3]

strategies_models = {
    'None': {
        'GaussianNB': GaussianNB(),
        'SVM': LinearSVC(random_state=seed, max_iter=2000),
        'Decision Tree': DecisionTreeClassifier(random_state=seed, max_depth=10),
        'Random Forest': RandomForestClassifier(random_state=seed, n_estimators=200, n_jobs=-1),
    },
    'Balanced': {
        'GaussianNB': GaussianNB(priors=[1/3, 1/3, 1/3]),
        'SVM': LinearSVC(random_state=seed, max_iter=2000, class_weight='balanced'),
        'Random Forest': RandomForestClassifier(random_state=seed, n_estimators=200, n_jobs=-1, class_weight='balanced'),
    },
    'SMOTE': {
        'GaussianNB': GaussianNB(),
        'SVM': LinearSVC(random_state=seed, max_iter=2000),
        'Decision Tree': DecisionTreeClassifier(random_state=seed, max_depth=10),
        'Random Forest': RandomForestClassifier(random_state=seed, n_estimators=200, n_jobs=-1),
    },
}

## Entrainement multi-stratégies

In [ ]:
results = []
trained = {}

for strategy, models_dict in strategies_models.items():

    X_tr = X_tr_smote if strategy == 'SMOTE' else X_train_proc
    y_tr = y_tr_smote if strategy == 'SMOTE' else y_train

    trained[strategy] = {}
    
    for k in pca_sizes:
        
        if k is not None:
            X_tr_pca = fit_transform_embedding(X_tr, method="pca", n_components=k)
            X_val_pca = fit_transform_embedding(X_val_proc, method="pca", n_components=k)
            
        else:
            X_tr_pca = X_tr
            X_val_pca = X_val_proc

        trained[strategy][k], result_tmp = train_and_eval(models_dict, X_tr_pca, y_tr, X_val_pca, y_val, task_type='classification', strategy=strategy, k=k, verbose=False)
        print(f"Done :      strat = {strategy}, k = {k}")
        results.extend(result_tmp)
        


Done :      strat = None, k = None
Done :      strat = None, k = 50
Done :      strat = None, k = 20
Done :      strat = None, k = 10
Done :      strat = None, k = 5
Done :      strat = None, k = 3
Done :      strat = Balanced, k = None
Done :      strat = Balanced, k = 50
Done :      strat = Balanced, k = 20
Done :      strat = Balanced, k = 10
Done :      strat = Balanced, k = 5
Done :      strat = Balanced, k = 3


### Résumé de l'entrainement

In [22]:
results_df = pd.DataFrame(results)
results_df['PCA-k'] = results_df['PCA-k'].astype('Int64')

print("=== Comparaison strategies x modeles ===")
print(results_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

pivot = results_df.pivot(index='Model', columns=['Strategy','PCA-k'], values='F1 macro')

print("\n=== Pivot : F1 macro par modele x strategie x PCA-k ===")
print(pivot.to_string(index=False, float_format=lambda x: f'{x:.4f}'))


=== Comparaison strategies x modeles ===
Strategy  PCA-k      Model  Accuracy  Balanced Acc  F1 macro  F1 Grave
    None   <NA> GaussianNB    0.0696        0.3573    0.0604    0.0197
    None     50 GaussianNB    0.6505        0.3949    0.3773    0.0236
    None     20 GaussianNB    0.6962        0.4047    0.3988    0.0252
    None     10 GaussianNB    0.7130        0.4339    0.4193    0.0093
    None      5 GaussianNB    0.7263        0.4485    0.4318    0.0096
    None      3 GaussianNB    0.5997        0.4107    0.3684    0.0000
Balanced   <NA> GaussianNB    0.0683        0.3570    0.0598    0.0197
Balanced     50 GaussianNB    0.4715        0.4209    0.3293    0.0259
Balanced     20 GaussianNB    0.4661        0.4568    0.3301    0.0541
Balanced     10 GaussianNB    0.4205        0.4784    0.3094    0.0520
Balanced      5 GaussianNB    0.4047        0.4938    0.2991    0.0649
Balanced      3 GaussianNB    0.4389        0.4054    0.2968    0.0028
   SMOTE   <NA> GaussianNB    0.1371

## Graphique de toutes combinaisons

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 10))

labels = ['Grave', 'Leger', 'Materiel']
model_names = ['GaussianNB', 'SVM', 'Random Forest']

for i, model_name in enumerate(model_names):
    for j, strategy in enumerate(strategies_models.keys()):

        ax = axes[i, j]
        model = trained[strategy][model_name]

        y_pred = model.predict(X_val_proc)

        cm = confusion_matrix(y_val, y_pred, labels=labels)

        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            ax=ax,
            xticklabels=labels,
            yticklabels=labels
        )

        ax.set_title(f"{model_name} - {strategy}")

plt.tight_layout()
plt.show()